### Clean Events (dedupe + normalization)

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
bronze_table = "subscription_platform.bronze.subscription_events"

silver_events_clean_table_path = "s3://subscription-revenue-platform/silver/subscription_events_clean/"
silver_events_clean_checkpoint_path = "s3://subscription-revenue-platform/silver/silver_events_clean_checkpoints/"
silver_events_clean_schema_path = f"{silver_events_clean_checkpoint_path}/schema"

In [0]:
catalog = "subscription_platform"
schema = "silver"
silver_events_clean_table = f"{catalog}.{schema}.subscription_events_clean"

In [0]:
bronze_df = (
    spark.readStream
    .option("schemaTrackingLocation", silver_events_clean_schema_path)
    .table(bronze_table)
)

In [0]:
events_clean = (
    bronze_df
    .withWatermark("event_time", "2 days")
    .dropDuplicates(["event_id"])
)

In [0]:
events_clean = (
    events_clean
    .withColumn("event_type", lower(col("event_type")))
    .withColumn("event_date", to_date("event_time"))
)

In [0]:
events_clean = events_clean.withColumn(
    "status",
    when(col("event_type") == "subscription_created", "active")
    .when(col("event_type") == "plan_changed", "active")
    .when(col("event_type") == "subscription_reactivated", "active")
    .when(col("event_type") == "subscription_cancelled", "cancelled")
)

In [0]:
silver_stream = (
    events_clean.writeStream
    .format("delta")
    .option("checkpointLocation", silver_events_clean_checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable(silver_events_clean_table)
)

In [0]:
silver_stream.awaitTermination()

In [0]:
%sql

SELECT *
FROM subscription_platform.silver.subscription_events_clean
LIMIT 20;

event_id,event_type,customer_id,subscription_id,plan_id,amount,currency,event_time,ingested_at,source,processed_at,source_file,event_date,status
evt_bc7ff48d9e6c4ddba2a3cc66f76ce295,subscription_cancelled,cust_509,sub_1009,basic,1000.0,INR,2026-04-01T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-01/evt_bc7ff48d9e6c4ddba2a3cc66f76ce295.json,2026-04-01,cancelled
evt_9b9f5d6a5beb42d285cdaa0e85d3ef94,subscription_cancelled,cust_507,sub_1007,basic,1000.0,INR,2026-04-01T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-01/evt_9b9f5d6a5beb42d285cdaa0e85d3ef94.json,2026-04-01,cancelled
evt_2469bb5c39334418a064baf440a455ec,subscription_cancelled,cust_504,sub_1004,basic,1000.0,INR,2026-04-01T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-01/evt_2469bb5c39334418a064baf440a455ec.json,2026-04-01,cancelled
evt_174b1bc7aede455ab83cc097eb00b29a,subscription_reactivated,cust_503,sub_1003,basic,1000.0,INR,2026-03-31T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-03-31/evt_174b1bc7aede455ab83cc097eb00b29a.json,2026-03-31,active
evt_190128ca6d674907ad8227f31a7b7bc1,subscription_cancelled,cust_505,sub_1005,basic,1000.0,INR,2026-04-01T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-01/evt_190128ca6d674907ad8227f31a7b7bc1.json,2026-04-01,cancelled
evt_9ce17eee3b3a4333a5ccd183e2880d40,subscription_reactivated,cust_509,sub_1009,basic,1000.0,INR,2026-03-31T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-03-31/evt_9ce17eee3b3a4333a5ccd183e2880d40.json,2026-03-31,active
evt_d7f63fead0ef44068dba7818f5ce8720,subscription_reactivated,cust_506,sub_1006,pro,2000.0,INR,2026-04-02T00:00:00.000Z,2026-04-02T00:00:00.000Z,billing_lambda,2026-04-27T18:18:03.901Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-02/evt_d7f63fead0ef44068dba7818f5ce8720.json,2026-04-02,active
evt_ba5e7a6672f841f3abebfbc7429e9eef,subscription_reactivated,cust_503,sub_1003,basic,1000.0,INR,2026-03-29T00:00:00.000Z,2026-04-01T00:00:00.000Z,billing_lambda,2026-04-24T19:12:30.188Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-03-29/evt_ba5e7a6672f841f3abebfbc7429e9eef.json,2026-03-29,active
evt_458fb75e166b4d9a9fe910237e6e4c6e,subscription_created,cust_501,sub_1001,basic,1000.0,INR,2026-04-02T00:00:00.000Z,2026-04-02T00:00:00.000Z,billing_lambda,2026-04-27T18:18:03.901Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-02/evt_458fb75e166b4d9a9fe910237e6e4c6e.json,2026-04-02,active
evt_5c1984c6f2fb4579aacec51895b15e8a,subscription_reactivated,cust_507,sub_1007,basic,1000.0,INR,2026-04-02T00:00:00.000Z,2026-04-02T00:00:00.000Z,billing_lambda,2026-04-27T18:18:03.901Z,s3://subscription-revenue-platform/raw/subscription_events/event_date=2026-04-02/evt_5c1984c6f2fb4579aacec51895b15e8a.json,2026-04-02,active


### Build Subscription State Table

In [0]:
from delta.tables import DeltaTable

In [0]:
silver_subscriptions_history_table_path = "s3://subscription-revenue-platform/silver/subscriptions_history/"
silver_subscriptions_history_checkpoint_path = "s3://subscription-revenue-platform/silver/subscriptions_history_checkpoints/"
silver_subscriptions_history_schema_path = f"{silver_subscriptions_history_checkpoint_path}/schema"

In [0]:
catalog = "subscription_platform"
schema = "silver"
silver_subscriptions_history_table = f"{catalog}.{schema}.subscriptions_history"

In [0]:
events_df = spark.read.table(silver_events_clean_table)

In [0]:
window_spec = Window.partitionBy("subscription_id").orderBy("event_time")

ordered_events = (
    events_df
    .withColumn("next_event_time", lead("event_time").over(window_spec))
)

In [0]:
history_df = (
    ordered_events
    .select(
        "subscription_id",
        "customer_id",
        "plan_id",
        "amount",
        "status",
        col("event_time").alias("start_date"),
        col("next_event_time").alias("end_date")
    )
)

In [0]:
history_df = history_df.withColumn(
    "is_current",
    col("end_date").isNull()
)

In [0]:
# Deduplicate source: keep the latest event per (subscription_id, start_date)
dedup_window = Window.partitionBy("subscription_id", "start_date").orderBy(col("end_date").desc_nulls_first())

In [0]:
history_deduped = (
    history_df
    .withColumn("_rank", row_number().over(dedup_window))
    .filter(col("_rank") == 1)
    .drop("_rank")
)


In [0]:
target = DeltaTable.forName(
    spark,
    "subscription_platform.silver.subscriptions_history"
)

(
    target.alias("target")
    .merge(
        history_deduped.alias("source"),
        """
        target.subscription_id = source.subscription_id
        AND target.start_date = source.start_date
        """
    )
    .whenMatchedUpdate(set={
        "end_date": "source.end_date",
        "is_current": "source.is_current",
        "plan_id": "source.plan_id",
        "amount": "source.amount",
        "status": "source.status"
    })
    .whenNotMatchedInsertAll()
    .execute()
)


DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql

SELECT *
FROM subscription_platform.silver.subscriptions_history
ORDER BY subscription_id, start_date;

subscription_id,customer_id,plan_id,amount,status,start_date,end_date,is_current
sub_1001,cust_501,basic,1000.0,active,2026-04-02T00:00:00.000Z,2026-04-03T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-03T00:00:00.000Z,2026-04-04T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-04T00:00:00.000Z,2026-04-05T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-05T00:00:00.000Z,2026-04-06T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-06T00:00:00.000Z,2026-04-07T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-07T00:00:00.000Z,2026-04-08T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-08T00:00:00.000Z,2026-04-09T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-09T00:00:00.000Z,2026-04-10T00:00:00.000Z,false
sub_1001,cust_501,pro,2000.0,active,2026-04-10T00:00:00.000Z,2026-04-11T00:00:00.000Z,false
sub_1001,cust_501,basic,1000.0,active,2026-04-11T00:00:00.000Z,2026-04-12T00:00:00.000Z,false
